# 🧥 Fashion Garment Classification — Fine-Tuning Notebook

**Goal**: Fine-tune an EfficientNet-B0 classifier on the **DeepFashion2** dataset to classify clothing into **15 categories**. Compare performance **with vs without** occlusion-aware data augmentation.

**Output**: `classifier.pth` — drop this into your `models/weights/` directory to replace the heuristic fallback.

---

### 📋 Sections
1. Setup & Dependencies
2. Dataset Download & Preparation
3. Augmentation Pipeline
4. Training — Baseline (No Augmentation)
5. Training — With Augmentation
6. Comparative Visualizations (10+ plots)
7. Export to Production

## 1️⃣ Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision timm ultralytics albumentations kagglehub \
    matplotlib seaborn scikit-learn tqdm pillow opencv-python-headless

In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# 15 clothing categories (must match app/config.py)
CLOTHING_CATEGORIES = [
    'shirt', 't-shirt', 'jacket', 'coat', 'sweater',
    'hoodie', 'jeans', 'pants', 'shorts', 'dress',
    'skirt', 'blouse', 'suit', 'tank_top', 'other',
]
NUM_CLASSES = len(CLOTHING_CATEGORIES)
print(f'Number of classes: {NUM_CLASSES}')
print(f'Categories: {CLOTHING_CATEGORIES}')

## 2️⃣ Dataset Download & Preparation

We use the **DeepFashion2** dataset. The dataset has 13 garment categories which we'll map to our 15-class schema.

**DeepFashion2 categories → Our categories:**
| DeepFashion2 | Our Label |
|---|---|
| short_sleeve_top | t-shirt |
| long_sleeve_top | shirt |
| short_sleeve_outwear | jacket |
| long_sleeve_outwear | coat |
| vest | tank_top |
| sling | tank_top |
| shorts | shorts |
| trousers | pants |
| skirt | skirt |
| short_sleeve_dress | dress |
| long_sleeve_dress | dress |
| vest_dress | dress |
| sling_dress | dress |

In [ ]:
# Download DeepFashion2 via kagglehub
import kagglehub

print('Downloading DeepFashion2 dataset (this may take a while)...')
dataset_path = kagglehub.dataset_download('pengyu/deepfashion2')
print(f'Dataset downloaded to: {dataset_path}')

In [ ]:
# Category mapping from DeepFashion2 to our schema
# DeepFashion2 uses integer category IDs (1-13)
DF2_TO_OURS = {
    1: 'shirt',       # short_sleeve_top -> shirt (close match)
    2: 'shirt',       # long_sleeve_top -> shirt
    3: 'jacket',      # short_sleeve_outwear -> jacket
    4: 'coat',        # long_sleeve_outwear -> coat
    5: 'tank_top',    # vest -> tank_top
    6: 'tank_top',    # sling -> tank_top
    7: 'shorts',      # shorts
    8: 'pants',       # trousers -> pants
    9: 'skirt',       # skirt
    10: 'dress',      # short_sleeve_dress -> dress
    11: 'dress',      # long_sleeve_dress -> dress
    12: 'dress',      # vest_dress -> dress
    13: 'dress',      # sling_dress -> dress
}

# Reverse: our label -> integer index (for training)
LABEL_TO_IDX = {lbl: i for i, lbl in enumerate(CLOTHING_CATEGORIES)}

print('Category mapping ready.')
print(f'Label → Index: {LABEL_TO_IDX}')

In [ ]:
# Parse DeepFashion2 annotations and crop garments
# We'll take images from the 'train' and 'validation' splits

DATA_ROOT = Path(dataset_path)
OUTPUT_DIR = Path('fashion_crops')
OUTPUT_DIR.mkdir(exist_ok=True)

# Create class subdirectories
for cat in CLOTHING_CATEGORIES:
    (OUTPUT_DIR / cat).mkdir(exist_ok=True)

def extract_crops(split='train', max_per_class=1500):
    """Extract clothing crops from DeepFashion2 annotations."""
    split_dir = DATA_ROOT / split
    image_dir = split_dir / 'image'
    annot_dir = split_dir / 'annos'
    
    if not image_dir.exists():
        # Try alternative structure
        image_dir = split_dir / 'images'
        annot_dir = split_dir / 'annotations'
    
    if not image_dir.exists():
        print(f'Warning: {image_dir} not found. Checking root...')
        # Walk the directory to find images
        for p in DATA_ROOT.rglob('*.jpg'):
            print(f'Found sample image: {p}')
            break
        return
    
    class_counts = Counter()
    annot_files = sorted(annot_dir.glob('*.json'))
    
    for annot_path in tqdm(annot_files, desc=f'Processing {split}'):
        img_name = annot_path.stem + '.jpg'
        img_path = image_dir / img_name
        if not img_path.exists():
            continue
        
        try:
            with open(annot_path) as f:
                annot = json.load(f)
        except Exception:
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        # Each annotation can have multiple items
        for key, item in annot.items():
            if not isinstance(item, dict):
                continue
            cat_id = item.get('category_id')
            if cat_id not in DF2_TO_OURS:
                continue
            
            our_label = DF2_TO_OURS[cat_id]
            if class_counts[our_label] >= max_per_class:
                continue
            
            # Get bounding box
            bbox = item.get('bounding_box')
            if not bbox or len(bbox) < 4:
                continue
            
            x1, y1, x2, y2 = map(int, bbox[:4])
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
            
            if (x2 - x1) < 32 or (y2 - y1) < 32:
                continue
            
            crop = img[y1:y2, x1:x2]
            crop_name = f'{annot_path.stem}_{key}.jpg'
            cv2.imwrite(str(OUTPUT_DIR / our_label / crop_name), crop)
            class_counts[our_label] += 1
    
    return class_counts

# Extract from both splits
print('Extracting training crops...')
train_counts = extract_crops('train', max_per_class=1500)
print(f'\nTrain crops per class: {train_counts}')

print('\nExtracting validation crops...')
val_counts = extract_crops('validation', max_per_class=500)
print(f'\nValidation crops per class: {val_counts}')

In [ ]:
# Verify dataset distribution
total_counts = Counter()
for cat in CLOTHING_CATEGORIES:
    count = len(list((OUTPUT_DIR / cat).glob('*.jpg')))
    total_counts[cat] = count

print('\n📊 Dataset Distribution:')
for cat, count in sorted(total_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (count // 50)
    print(f'  {cat:12s} | {count:5d} | {bar}')
print(f'  {"TOTAL":12s} | {sum(total_counts.values()):5d}')

In [ ]:
# Split into train / val / test (70 / 15 / 15)
from sklearn.model_selection import train_test_split

SPLIT_DIR = Path('fashion_split')
for split in ['train', 'val', 'test']:
    for cat in CLOTHING_CATEGORIES:
        (SPLIT_DIR / split / cat).mkdir(parents=True, exist_ok=True)

split_counts = {'train': 0, 'val': 0, 'test': 0}

for cat in CLOTHING_CATEGORIES:
    images = sorted((OUTPUT_DIR / cat).glob('*.jpg'))
    if len(images) < 3:
        print(f'  ⚠️ Skipping {cat}: only {len(images)} images')
        continue
    
    # 70/15/15 split
    train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=SEED)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=SEED)
    
    for img_path in train_imgs:
        shutil.copy2(img_path, SPLIT_DIR / 'train' / cat / img_path.name)
    for img_path in val_imgs:
        shutil.copy2(img_path, SPLIT_DIR / 'val' / cat / img_path.name)
    for img_path in test_imgs:
        shutil.copy2(img_path, SPLIT_DIR / 'test' / cat / img_path.name)
    
    split_counts['train'] += len(train_imgs)
    split_counts['val'] += len(val_imgs)
    split_counts['test'] += len(test_imgs)

print('✅ Dataset split complete:')
for split, count in split_counts.items():
    print(f'  {split}: {count} images')

## 3️⃣ Data Augmentation Pipeline

Apply standard augmentations (flips, rotations, colour jitter) for the augmented training set. This simulates the occlusion-aware pipeline from the project.

In [ ]:
import albumentations as A

# Standard augmentation transforms
augment_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.RandomScale(scale_limit=0.2, p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.4),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
    A.GaussianBlur(blur_limit=3, p=0.2),
    A.Perspective(scale=(0.02, 0.05), p=0.3),
    A.CoarseDropout(max_holes=3, max_height=32, max_width=32, p=0.2),  # Simulates occlusion
])

# Generate augmented dataset (5x the training set)
AUG_DIR = Path('fashion_split') / 'train_augmented'

aug_count = 0
for cat in tqdm(CLOTHING_CATEGORIES, desc='Augmenting'):
    src_dir = SPLIT_DIR / 'train' / cat
    dst_dir = AUG_DIR / cat
    dst_dir.mkdir(parents=True, exist_ok=True)
    
    images = list(src_dir.glob('*.jpg'))
    
    # Copy originals
    for img_path in images:
        shutil.copy2(img_path, dst_dir / img_path.name)
        aug_count += 1
    
    # Generate 4 augmented versions per original
    for img_path in images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        for aug_i in range(4):
            augmented = augment_transform(image=img_rgb)['image']
            aug_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
            aug_name = f'{img_path.stem}_aug{aug_i}.jpg'
            cv2.imwrite(str(dst_dir / aug_name), aug_bgr)
            aug_count += 1

print(f'\n✅ Augmented dataset: {aug_count} total images')

In [ ]:
# Visualize augmentation samples
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
fig.suptitle('Augmentation Samples — Original vs Augmented', fontsize=16, fontweight='bold')

sample_cats = random.sample(CLOTHING_CATEGORIES, min(3, len(CLOTHING_CATEGORIES)))

for row, cat in enumerate(sample_cats):
    src_dir = SPLIT_DIR / 'train' / cat
    images = list(src_dir.glob('*.jpg'))
    if not images:
        continue
    
    sample_img = cv2.imread(str(images[0]))
    sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    
    axes[row][0].imshow(sample_rgb)
    axes[row][0].set_title(f'{cat} (Original)', fontsize=10)
    axes[row][0].axis('off')
    
    for col in range(1, 5):
        aug = augment_transform(image=sample_rgb)['image']
        axes[row][col].imshow(aug)
        axes[row][col].set_title(f'Aug {col}', fontsize=10)
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('augmentation_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: augmentation_gallery.png')

## 4️⃣ Dataset & Model Setup

In [ ]:
# Dataset class
class FashionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples = []
        self.labels = []
        
        root = Path(root_dir)
        for cat in CLOTHING_CATEGORIES:
            cat_dir = root / cat
            if not cat_dir.exists():
                continue
            idx = LABEL_TO_IDX[cat]
            for img_path in cat_dir.glob('*.jpg'):
                self.samples.append(str(img_path))
                self.labels.append(idx)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img = Image.open(self.samples[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Dataset and transform classes ready.')

In [ ]:
def create_model():
    """Create a fresh EfficientNet-B0 with custom head."""
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    return model.to(DEVICE)

# Training hyperparameters
EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4

print(f'Hyperparameters: epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LR}')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler=None):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    
    if scheduler:
        scheduler.step()
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return total_loss / total, correct / total, all_preds, all_labels

print('Training functions ready.')

## 5️⃣ Training — Baseline (No Augmentation)

In [ ]:
# Create dataloaders — BASELINE (no augmentation)
train_ds_base = FashionDataset(SPLIT_DIR / 'train', transform=train_transform)
val_ds = FashionDataset(SPLIT_DIR / 'val', transform=val_transform)
test_ds = FashionDataset(SPLIT_DIR / 'test', transform=val_transform)

train_loader_base = DataLoader(train_ds_base, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Baseline training: {len(train_ds_base)} images')
print(f'Validation: {len(val_ds)} images')
print(f'Test: {len(test_ds)} images')

In [ ]:
# Train baseline model
model_base = create_model()
criterion = nn.CrossEntropyLoss()
optimizer_base = optim.AdamW(model_base.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_base = optim.lr_scheduler.CosineAnnealingLR(optimizer_base, T_max=EPOCHS)

history_base = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_base = 0

print('🏋️ Training BASELINE model (no augmentation)...')
print('-' * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model_base, train_loader_base, criterion, optimizer_base, scheduler_base)
    val_loss, val_acc, _, _ = evaluate(model_base, val_loader, criterion)
    
    history_base['train_loss'].append(train_loss)
    history_base['train_acc'].append(train_acc)
    history_base['val_loss'].append(val_loss)
    history_base['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_base:
        best_val_acc_base = val_acc
        torch.save(model_base.state_dict(), 'baseline_best.pth')
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} {"✓" if val_acc == best_val_acc_base else ""}')

print(f'\n✅ Baseline best validation accuracy: {best_val_acc_base:.4f}')

## 6️⃣ Training — With Augmentation

In [ ]:
# Create dataloaders — WITH AUGMENTATION
train_ds_aug = FashionDataset(AUG_DIR, transform=train_transform)
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f'Augmented training: {len(train_ds_aug)} images (5x baseline)')

In [ ]:
# Train augmented model  
model_aug = create_model()
optimizer_aug = optim.AdamW(model_aug.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_aug = optim.lr_scheduler.CosineAnnealingLR(optimizer_aug, T_max=EPOCHS)

history_aug = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc_aug = 0

print('🏋️ Training AUGMENTED model...')
print('-' * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model_aug, train_loader_aug, criterion, optimizer_aug, scheduler_aug)
    val_loss, val_acc, _, _ = evaluate(model_aug, val_loader, criterion)
    
    history_aug['train_loss'].append(train_loss)
    history_aug['train_acc'].append(train_acc)
    history_aug['val_loss'].append(val_loss)
    history_aug['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc_aug:
        best_val_acc_aug = val_acc
        torch.save(model_aug.state_dict(), 'augmented_best.pth')
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} {"✓" if val_acc == best_val_acc_aug else ""}')

print(f'\n✅ Augmented best validation accuracy: {best_val_acc_aug:.4f}')
print(f'Improvement over baseline: {(best_val_acc_aug - best_val_acc_base)*100:+.2f}%')

## 7️⃣ Comparative Visualizations (10+ Plots)

In [ ]:
# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

epochs_range = range(1, EPOCHS + 1)

In [ ]:
# PLOT 1: Training Loss Curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, history_base['train_loss'], 'o-', label='Baseline', linewidth=2, markersize=4)
ax.plot(epochs_range, history_aug['train_loss'], 's-', label='Augmented', linewidth=2, markersize=4)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('📉 Training Loss: Baseline vs Augmented', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_01_train_loss.png', dpi=150)
plt.show()

In [ ]:
# PLOT 2: Validation Accuracy Curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, [a*100 for a in history_base['val_acc']], 'o-', label='Baseline', linewidth=2, markersize=4)
ax.plot(epochs_range, [a*100 for a in history_aug['val_acc']], 's-', label='Augmented', linewidth=2, markersize=4)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('📈 Validation Accuracy: Baseline vs Augmented', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_02_val_accuracy.png', dpi=150)
plt.show()

In [ ]:
# PLOT 3: Validation Loss Curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, history_base['val_loss'], 'o-', label='Baseline', linewidth=2, markersize=4)
ax.plot(epochs_range, history_aug['val_loss'], 's-', label='Augmented', linewidth=2, markersize=4)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title('📉 Validation Loss: Baseline vs Augmented', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_03_val_loss.png', dpi=150)
plt.show()

In [ ]:
# Evaluate both models on test set
model_base.load_state_dict(torch.load('baseline_best.pth', map_location=DEVICE))
model_aug.load_state_dict(torch.load('augmented_best.pth', map_location=DEVICE))

_, test_acc_base, preds_base, labels_test = evaluate(model_base, test_loader, criterion)
_, test_acc_aug, preds_aug, _ = evaluate(model_aug, test_loader, criterion)

print(f'Test Accuracy — Baseline:  {test_acc_base*100:.2f}%')
print(f'Test Accuracy — Augmented: {test_acc_aug*100:.2f}%')
print(f'Improvement: {(test_acc_aug - test_acc_base)*100:+.2f}%')

In [ ]:
# PLOT 4 & 5: Confusion Matrices (side by side)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

# Use only categories that appear in test set
present_classes = sorted(set(labels_test))
present_names = [CLOTHING_CATEGORIES[i] for i in present_classes]

cm_base = confusion_matrix(labels_test, preds_base, labels=present_classes)
cm_aug = confusion_matrix(labels_test, preds_aug, labels=present_classes)

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', xticklabels=present_names,
            yticklabels=present_names, ax=ax1, cbar_kws={'shrink': 0.8})
ax1.set_title(f'Baseline (Test Acc: {test_acc_base*100:.1f}%)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')

sns.heatmap(cm_aug, annot=True, fmt='d', cmap='Greens', xticklabels=present_names,
            yticklabels=present_names, ax=ax2, cbar_kws={'shrink': 0.8})
ax2.set_title(f'Augmented (Test Acc: {test_acc_aug*100:.1f}%)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')

fig.suptitle('🔍 Confusion Matrices — Baseline vs Augmented', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_04_05_confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# PLOT 6: Per-Class Accuracy Bar Chart
def per_class_accuracy(preds, labels, classes):
    acc = {}
    for cls in classes:
        mask = np.array(labels) == cls
        if mask.sum() == 0:
            continue
        acc[CLOTHING_CATEGORIES[cls]] = (np.array(preds)[mask] == cls).mean() * 100
    return acc

pca_base = per_class_accuracy(preds_base, labels_test, present_classes)
pca_aug = per_class_accuracy(preds_aug, labels_test, present_classes)

cats = list(pca_base.keys())
x = np.arange(len(cats))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - w/2, [pca_base.get(c, 0) for c in cats], w, label='Baseline', color='#60A5FA')
bars2 = ax.bar(x + w/2, [pca_aug.get(c, 0) for c in cats], w, label='Augmented', color='#34D399')

ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('📊 Per-Class Accuracy: Baseline vs Augmented', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=35, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2, axis='y')
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('plot_06_per_class_accuracy.png', dpi=150)
plt.show()

In [ ]:
# PLOT 7: Training Dashboard (4 subplots combined)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('📋 Training Dashboard', fontsize=16, fontweight='bold')

# Train loss
axes[0,0].plot(epochs_range, history_base['train_loss'], 'o-', label='Baseline', linewidth=1.5, markersize=3)
axes[0,0].plot(epochs_range, history_aug['train_loss'], 's-', label='Augmented', linewidth=1.5, markersize=3)
axes[0,0].set_title('Training Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Val loss
axes[0,1].plot(epochs_range, history_base['val_loss'], 'o-', label='Baseline', linewidth=1.5, markersize=3)
axes[0,1].plot(epochs_range, history_aug['val_loss'], 's-', label='Augmented', linewidth=1.5, markersize=3)
axes[0,1].set_title('Validation Loss'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# Train acc
axes[1,0].plot(epochs_range, [a*100 for a in history_base['train_acc']], 'o-', label='Baseline', linewidth=1.5, markersize=3)
axes[1,0].plot(epochs_range, [a*100 for a in history_aug['train_acc']], 's-', label='Augmented', linewidth=1.5, markersize=3)
axes[1,0].set_title('Training Accuracy (%)'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# Val acc
axes[1,1].plot(epochs_range, [a*100 for a in history_base['val_acc']], 'o-', label='Baseline', linewidth=1.5, markersize=3)
axes[1,1].plot(epochs_range, [a*100 for a in history_aug['val_acc']], 's-', label='Augmented', linewidth=1.5, markersize=3)
axes[1,1].set_title('Validation Accuracy (%)'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('plot_07_training_dashboard.png', dpi=150)
plt.show()

In [ ]:
# PLOT 8: ROC Curves (per class, augmented model)
# Get probabilities from augmented model
model_aug.eval()
all_probs = []
all_labels_for_roc = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model_aug(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels_for_roc.extend(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels_bin = label_binarize(all_labels_for_roc, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab20(np.linspace(0, 1, len(present_classes)))

for i, cls in enumerate(present_classes):
    if all_labels_bin[:, cls].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(all_labels_bin[:, cls], all_probs[:, cls])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=1.5,
            label=f'{CLOTHING_CATEGORIES[cls]} (AUC={roc_auc:.2f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('📊 ROC Curves per Class (Augmented Model)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_08_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# PLOT 9: t-SNE of Learned Embeddings (Augmented Model)
# Extract features from the penultimate layer
model_aug.eval()
features = []
tsne_labels = []

# Hook to get penultimate features
activation = {}
def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

# Register hook on global average pooling
hook_handle = model_aug.global_pool.register_forward_hook(get_activation('features'))

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        _ = model_aug(images)
        feats = activation['features'].cpu().numpy()
        if feats.ndim > 2:
            feats = feats.reshape(feats.shape[0], -1)
        features.append(feats)
        tsne_labels.extend(labels.numpy())

hook_handle.remove()
features = np.concatenate(features)

# Run t-SNE
print(f'Running t-SNE on {features.shape[0]} samples, {features.shape[1]} features...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
features_2d = tsne.fit_transform(features)

fig, ax = plt.subplots(figsize=(12, 9))
scatter = ax.scatter(features_2d[:, 0], features_2d[:, 1],
                     c=tsne_labels, cmap='tab20', alpha=0.6, s=15)

# Legend
handles = [plt.Line2D([0], [0], marker='o', color='w',
           markerfacecolor=plt.cm.tab20(i/NUM_CLASSES), markersize=8,
           label=CLOTHING_CATEGORIES[i]) for i in present_classes]
ax.legend(handles=handles, loc='best', fontsize=8, ncol=2)

ax.set_title('🔮 t-SNE of Learned Embeddings (Augmented Model)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('plot_09_tsne_embeddings.png', dpi=150)
plt.show()

In [ ]:
# PLOT 10: Top-5 Most Misclassified Images
model_aug.eval()
misclassified = []

with torch.no_grad():
    for idx in range(len(test_ds)):
        img, label = test_ds[idx]
        output = model_aug(img.unsqueeze(0).to(DEVICE))
        probs = torch.softmax(output, dim=1)[0]
        pred = probs.argmax().item()
        conf = probs[pred].item()
        
        if pred != label:
            misclassified.append({
                'idx': idx,
                'true': CLOTHING_CATEGORIES[label],
                'pred': CLOTHING_CATEGORIES[pred],
                'conf': conf,
                'path': test_ds.samples[idx],
            })

# Sort by confidence (most confidently wrong)
misclassified.sort(key=lambda x: -x['conf'])

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('❌ Top-5 Most Confidently Misclassified', fontsize=14, fontweight='bold')

for i, item in enumerate(misclassified[:5]):
    img = Image.open(item['path']).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f"True: {item['true']}\nPred: {item['pred']} ({item['conf']*100:.0f}%)",
                      fontsize=9, color='red')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('plot_10_misclassified.png', dpi=150)
plt.show()

In [ ]:
# PLOT 11: Sample Prediction Grid (Augmented Model)
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
fig.suptitle('✅ Sample Predictions (Augmented Model)', fontsize=16, fontweight='bold')

model_aug.eval()
sample_indices = random.sample(range(len(test_ds)), min(15, len(test_ds)))

for i, idx in enumerate(sample_indices):
    row, col = i // 5, i % 5
    img_tensor, true_label = test_ds[idx]
    
    with torch.no_grad():
        output = model_aug(img_tensor.unsqueeze(0).to(DEVICE))
        probs = torch.softmax(output, dim=1)[0]
        pred = probs.argmax().item()
        conf = probs[pred].item()
    
    # Load original image
    img = Image.open(test_ds.samples[idx]).convert('RGB')
    axes[row][col].imshow(img)
    
    correct = pred == true_label
    color = 'green' if correct else 'red'
    axes[row][col].set_title(
        f'{CLOTHING_CATEGORIES[pred]} ({conf*100:.0f}%)',
        fontsize=9, color=color, fontweight='bold'
    )
    axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('plot_11_sample_predictions.png', dpi=150)
plt.show()

In [ ]:
# PLOT 12: Summary Comparison Bar
fig, ax = plt.subplots(figsize=(8, 5))

metrics = ['Val Accuracy', 'Test Accuracy']
base_vals = [best_val_acc_base * 100, test_acc_base * 100]
aug_vals = [best_val_acc_aug * 100, test_acc_aug * 100]

x = np.arange(len(metrics))
w = 0.3

bars1 = ax.bar(x - w/2, base_vals, w, label='Baseline', color='#60A5FA')
bars2 = ax.bar(x + w/2, aug_vals, w, label='Augmented', color='#34D399')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('🏆 Final Comparison: Baseline vs Augmented', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.savefig('plot_12_final_comparison.png', dpi=150)
plt.show()

In [ ]:
# Print classification report
print('=' * 70)
print('CLASSIFICATION REPORT — Augmented Model (Test Set)')
print('=' * 70)
print(classification_report(
    labels_test, preds_aug,
    labels=present_classes,
    target_names=[CLOTHING_CATEGORIES[i] for i in present_classes],
    digits=3
))

## 8️⃣ Export to Production

Save the best augmented model as `classifier.pth`. Copy this to your project's `models/weights/` directory.

In [ ]:
# Save the best model for deployment
EXPORT_PATH = 'classifier.pth'

# Load best augmented checkpoint
model_export = create_model()
model_export.load_state_dict(torch.load('augmented_best.pth', map_location='cpu'))

# Save
torch.save(model_export.state_dict(), EXPORT_PATH)
file_size = os.path.getsize(EXPORT_PATH) / 1024 / 1024

print(f'✅ Model exported to: {EXPORT_PATH}')
print(f'   File size: {file_size:.1f} MB')
print(f'   Test accuracy: {test_acc_aug*100:.2f}%')
print(f'   Classes: {NUM_CLASSES}')
print()
print('📥 NEXT STEPS:')
print('   1. Download classifier.pth from Colab')
print('   2. Copy to: models/weights/classifier.pth in your project')
print('   3. Restart the server: uvicorn app.main:app --reload --port 8001')
print('   4. The model will now use fine-tuned weights instead of heuristics!')

In [ ]:
# Optional: save to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_path = '/content/drive/MyDrive/fashion_ai_models/'
    os.makedirs(drive_path, exist_ok=True)
    shutil.copy2(EXPORT_PATH, drive_path + 'classifier.pth')
    print(f'✅ Also saved to Google Drive: {drive_path}classifier.pth')
except Exception as e:
    print(f'Google Drive save skipped: {e}')